# 1. QUERY-DEPENDENT LAG-ADAPTIVE LACA — AeroSync-Mamba v21

v21 starts from v18 and tests a different lag mechanism.

**Background.** v17 (fixed β = 1), v18 (stabilized β = 1) and v19 (learnable β) all found that LACA's **single global**
lag-bias term β·g(Δτ) gives no real benefit. The learnable β collapsed toward 0 on every seed. v21 tests another
hypothesis: one global Δτ → bias function may be too crude, and **each query token may need its own preferred lag
and tolerance.**

**The mechanism.** It replaces β·g(Δτ) entirely.
- A `Linear(256, 2)` lag predictor per LACA direction reads each query token's representation H_i. This is the same
  H_m that feeds W_q.
- From it: **μ_i = tanh(·)·0.5** and **σ_i = softplus(·) + 0.02**, in normalized window time where Δτ ∈ [−1, 1].
- The bias added to the content scores before the softmax is **bias_ij = −½((Δτ_ij − μ_i)/σ_i)²**, where Δτ_ij =
  τ_query,i − τ_key,j (the same Δτ v18 used).
- There is no β and no global g(Δτ).
- No label or class information reaches the mechanism. μ and σ depend only on H_i.

**Initialization.** The predictor's weights are drawn **N(0, 0.001²)** and its bias is 0. At initialization the query
tokens H_i are nearly identical: they leave HTT's LayerNorm with norm about √256 = 16 and share most of their direction.
So w·H_i is essentially **one random offset per seed and direction**, with std about 0.001 · 16 = 0.016, plus a tiny
spread across tokens.

The first Kaggle run used std 0.01. There, seed 7 started with **μ ≈ −0.16 (−420 ms) on every audio query token**,
with a spread across tokens of only 0.0099. That is not tanh saturation, but it is an arbitrary, seed-specific starting
lag, which is exactly what this experiment must avoid. **Expected at 0.001:**
- **μ:** about 0 ± 0.008 per seed (about 20 ms).
- **σ:** about softplus(0) + 0.02 = **0.713**.

These are calculated from the formulas, not measured. Section 5 **measures** them on a real batch for every seed
before any training step. It stops the notebook if the starting lag |μ| averages above 0.05 (about 128 ms), if any
|μ| reaches tanh saturation (0.45), or if σ is extreme. Unlike v18's zero-init, the lag
term is **not** 0 at initialization. It starts as a broad Gaussian preference for small |Δτ − μ|, at most about −1.4
at |Δτ| = 1.

**Keeping the pairing exact.** v18 built g(Δτ) inside each LACA direction. v21 still draws those weights and throws
them away (the same idea as `RNG_REFERENCE_STACK`), and initializes the lag predictor from a *forked* RNG. So every
shared layer starts from exactly the control's initial weights, and the shuffle and dropout stream is identical. The
torch RNG state after construction is checked against the control's for every seed.

**Control: v18's five β = 0.0 runs, reused, not retrained.** Macro-F1 0.9057 / 0.8827 / 0.8764 / 0.9019 / 0.8638. The
4 metrics come from v18's CSV. The confusion matrices, which give Class 3/4 recall and the 3→4 / 4→3 counts, come from
v20, which regenerated this exact control and reproduced v18's 4 metrics exactly. Both are embedded in Section 2 and
cross-checked there.

**Part A.** Train the new architecture on 5 seeds. Paired statistics against the control on macro-F1 and Class 4
recall. Reported with equal care: Class 3/4 recall, the 3→4 and 4→3 counts, and the **learned μ/σ distributions**.

**Part B.** Timestamp-shuffle control on the same 5 trained models, with no retraining. In every test window the
128 sensor timestamps are permuted among themselves while the sensor values stay bit-identical. Two scopes are
evaluated:
- **all**: the shuffled τ_s is used everywhere, i.e. HTT's time embedding as well as LACA. This is the requested test.
- **LACA only**: HTT sees the true τ_s, and only the new lag term sees the shuffled one. This isolates the new
  mechanism.

If v18's or v20's control checkpoints are attached, the control is shuffled too. That separates what the new
mechanism adds from what HTT already does with timestamps.

| Decision | Value | Evidence status |
|---|---|---|
| Timestamps into HTT / LACA | Real per-window audio and IMU timestamps (not a synthetic `linspace`) | CORRECTNESS FIX (earlier versions passed an identical synthetic grid for every window) |
| Fusion backbone | Real `mamba_ssm.Mamba` selective-scan SSM; pure-PyTorch fallback only when no CUDA GPU is available (printed unmissably) | CORRECTNESS FIX (the earlier gated-conv block was not an SSM) |
| Audio bin selection | Strongest-N FFT bins by mean magnitude, computed once from **training** audio only, then frozen and reused for val/test | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): strongest-64 0.8736 vs first-64 0.8494, +0.0242 (bin-selection study) |
| Frequency budget N | 64 bins | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v6: N=64 0.8966 vs N=48 0.8863, **+0.0103**, which is below 0.0160. The other budgets clear it (N=32 +0.0249, N=128 +0.0166) except N=96 (+0.0142). |
| Audio compression | `sqrt(|rFFT|)` | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 sqrt 0.8966 vs log_db 0.8696 (+0.0270) and log1p 0.8649 (+0.0317) |
| Audio windowing | None; Hann windowing rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 no window 0.8649 vs Hann 0.8290 (+0.0359) |
| Audio normalization | None; z-score normalization rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 none 0.8966 vs z-score 0.8684 (+0.0282) |
| Audio time positions Ta | 64 per 2.56 s window | PROVISIONAL: v7 Ta=128 0.8967 vs Ta=64 0.8966 (−0.0001, a tie). The cheaper Ta=64 is kept. |
| Sensor input | Raw 6 channels (accel x/y/z, gyro x/y/z), Ts=128; derived magnitude features rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v8 raw6 0.8966 vs best derived set 0.8740 (+0.0226) |
| LACA global lag bias β·g(Δτ) | **Removed in v21** (control = v18 at β = 0) | NOT ESTABLISHED: v17 +0.0008 ± 0.0020 (3/5), v18 −0.0094 ± 0.0136 (1/5), v19 learnable β collapsed toward 0 on all 5 seeds. |
| LACA lag mechanism | **v21: query-dependent**, per query token μ_i = tanh(·)·0.5, σ_i = softplus(·) + 0.02 from `Linear(256, 2)`; bias −½((Δτ_ij − μ_i)/σ_i)² | **UNDER TEST IN THIS NOTEBOOK** (paired 5 seeds vs the reused β = 0 control, plus a timestamp-shuffle control). |
| LACA heads | 4 | PROVISIONAL: v11 h=1 was +0.0008 (one test window) and h=4 was kept. h=2/8/16 were 0.006–0.012 worse (below 0.0160). |
| Mamba fusion | 2 blocks, d_state=16, d_conv=2 (expand 2) | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v12: 2 vs 4 blocks +0.0023, d_state 16 vs 8 +0.0172, d_conv 2 vs 4 +0.0059. |
| λ_align (InfoNCE) | 0.1 | PROVISIONAL: v13 removing L_align cost −0.0088, which is below 0.0160. λ=0.05 was −0.0271 and λ=0.2 was −0.0070. |
| λ_cons (KL consistency) | 0.1 | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v13 grid: λ=0.2 +0.0008, λ=0.05 −0.0232. Removing L_cons cost −0.0165. |
| Ordinal-severity loss | Not used (rejected) | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v14 best λ_ord>0 0.8859 vs none 0.9048 (−0.0189); adjacent-class errors rose 94 → 113 |
| Classification loss | Cross-entropy weighted by inverse **training-set** class frequency | CORRECTNESS FIX (class-imbalance handling) |
| Seeding | Python `random`, numpy, torch CPU + all CUDA devices, cuDNN deterministic; reset per seed at run start and right before model construction. The mission split has its own fixed seed (42), so every seed sees the same train/val/test windows. | CORRECTNESS FIX (reproducibility) |

## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [1]:
import copy
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table above)
# ==============================================================================
# --- Query-dependent lag LACA: one run per seed (the control is REUSED from v18, not retrained) ---
SEEDS = [42, 123, 2024, 7, 99]           # same 5 seeds as every prior multi-seed experiment
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LAG_BIAS_HIDDEN = 32                     # v18's g(Δτ) width — v21 only DRAWS and discards it (RNG alignment), never uses it
# --- v21: query-dependent lag bias, per query token i: bias_ij = -0.5 * ((Δτ_ij - mu_i) / sigma_i)^2 ---
MAX_LAG = 0.5                            # mu_i = tanh(lag_predictor(H_i)[0]) * MAX_LAG   (normalized window time)
MIN_SIGMA = 0.02                         # sigma_i = softplus(lag_predictor(H_i)[1]) + MIN_SIGMA
LAG_PREDICTOR_INIT_STD = 0.001           # final (only) Linear(256, 2): weights N(0, 0.001^2), bias 0. At init the query
                                         # tokens are nearly identical, so w.H_i is ~ONE random offset per seed with std
                                         # ~= std * 16; at 0.01 that gave seed 7 mu ~ -0.16 (-420 ms). At 0.001: offset std
                                         # ~0.016 -> |mu| ~0.008 typical (~20 ms), still fully trainable (linear layer).
INIT_MU_ABSMEAN_MAX = 0.05               # init sanity: mean |mu| (the per-seed starting lag) must stay below this (~128 ms,
                                         # ~6x the expected per-seed offset at this init scale) ...
INIT_MU_SATURATION = 0.9 * MAX_LAG       # ... and no |mu| may reach this (tanh saturation)
INIT_SIGMA_RANGE = (0.3, 1.5)            # init sanity: every sigma inside this range (expected ~0.713)
HALT_ON_BAD_LAG_INIT = True              # stop instead of training from a saturated / extreme init
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- Part B: timestamp-shuffle control (evaluation only, same trained models) ---
SHUFFLE_SEED = 2718                      # private generator for the per-window permutations of tau_s (global RNG untouched)
SHUFFLE_MATERIAL_F1 = 0.005              # a degradation below this (~6 test windows) is treated as negligible
SHUFFLE_MATERIAL_C4 = 0.01               # same for Class 4 recall (2 of its 201 test windows)

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v21_checkpoints"
PROBS_DIR = WORK_DIR / "v21_test_probs"
RESULTS_CSV = WORK_DIR / "query_lag_laca_results.csv"
SHUFFLE_CSV = WORK_DIR / "query_lag_laca_shuffle_results.csv"
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- CONTROL: v18's beta=0.0 runs, REUSED (not retrained) ---
# 4 metrics + init/RNG fingerprints: Artifacts/Version 18 CSV. Confusion matrices: Artifacts/Version 20 CSV, whose
# control rows are this same control regenerated in v20 (they reproduced v18's 4 metrics exactly).
V18_BETA0_CONTROL = {
    42: {"accuracy": 0.9088495575221239, "macro_f1": 0.9057330017972403, "balanced_accuracy": 0.9030033658152373, "macro_auroc": 0.9891867537737307,
          "confusion_matrix": [[283, 0, 1, 0, 2], [3, 197, 2, 1, 2], [0, 1, 172, 35, 2], [0, 0, 8, 204, 16], [0, 0, 2, 28, 171]],
          "init_sha1": "5850841e699f", "rng_sha1": "5641b401720d"},
    123: {"accuracy": 0.8876106194690265, "macro_f1": 0.8826825575079713, "balanced_accuracy": 0.8790467961967972, "macro_auroc": 0.9861866752615566,
          "confusion_matrix": [[283, 1, 1, 0, 1], [2, 196, 6, 1, 0], [0, 2, 177, 30, 1], [0, 0, 8, 203, 17], [0, 1, 2, 54, 144]],
          "init_sha1": "74be0883c62b", "rng_sha1": "0a4b3be44b18"},
    2024: {"accuracy": 0.8805309734513275, "macro_f1": 0.8763643000413779, "balanced_accuracy": 0.8715485268344848, "macro_auroc": 0.9866958753709998,
          "confusion_matrix": [[282, 1, 0, 0, 3], [2, 189, 12, 1, 1], [0, 0, 168, 39, 3], [0, 0, 6, 206, 16], [0, 0, 0, 51, 150]],
          "init_sha1": "92852955eef1", "rng_sha1": "619c0e0a6b4d"},
    7: {"accuracy": 0.9061946902654867, "macro_f1": 0.9019350016739314, "balanced_accuracy": 0.9008047135870789, "macro_auroc": 0.9897445660698647,
          "confusion_matrix": [[284, 1, 0, 0, 1], [2, 196, 7, 0, 0], [0, 3, 183, 22, 2], [0, 0, 16, 191, 21], [0, 0, 3, 28, 170]],
          "init_sha1": "7f5d0cd2e7cb", "rng_sha1": "9fdcf777d285"},
    99: {"accuracy": 0.8690265486725663, "macro_f1": 0.8638026478536822, "balanced_accuracy": 0.8598248510633313, "macro_auroc": 0.9829841151452052,
          "confusion_matrix": [[282, 0, 1, 0, 3], [5, 190, 9, 0, 1], [0, 2, 164, 39, 5], [0, 0, 12, 197, 19], [0, 1, 3, 48, 149]],
          "init_sha1": "e40c78169f57", "rng_sha1": "81dcd4638a88"},
}
# Optional, Part B only: the control's trained weights (attach v18's or v20's notebook output as a Kaggle input) so
# the control can be timestamp-shuffled too. Not needed for Part A.
CONTROL_CKPT_GLOBS = ["/kaggle/input/**/v18_checkpoints/beta0.0_seed{seed}_best.pt",
                      "/kaggle/input/**/v20_checkpoints/ce_seed{seed}_best.pt"]
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1
PAIRED_ALPHA = 0.05
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v15)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"QUERY-DEPENDENT LAG LACA: {len(SEEDS)} runs (seeds {SEEDS}) vs the REUSED v18 beta=0.0 control (not retrained) "
      f"| Part A -> {RESULTS_CSV} | Part B (timestamp shuffle) -> {SHUFFLE_CSV}")
print(f"per query token: mu = tanh(.) * {MAX_LAG}, sigma = softplus(.) + {MIN_SIGMA}, bias = -0.5 ((Δτ - mu) / sigma)^2 | "
      f"lag predictor init N(0, {LAG_PREDICTOR_INIT_STD}^2), bias 0 | no beta, no global g(Δτ)")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA query-dependent lag (mu_i, sigma_i), {LACA_NUM_HEADS} heads "
      f"| Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | "
      f"loss CE + {LAMBDA_ALIGN} InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")

Device: cuda (Tesla T4)
QUERY-DEPENDENT LAG LACA: 5 runs (seeds [42, 123, 2024, 7, 99]) vs the REUSED v18 beta=0.0 control (not retrained) | Part A -> /kaggle/working/query_lag_laca_results.csv | Part B (timestamp shuffle) -> /kaggle/working/query_lag_laca_shuffle_results.csv
per query token: mu = tanh(.) * 0.5, sigma = softplus(.) + 0.02, bias = -0.5 ((Δτ - mu) / sigma)^2 | lag predictor init N(0, 0.001^2), bias 0 | no beta, no global g(Δτ)
Fixed configuration: audio strongest-64, Ta=64, sqrt, no window, no normalization | sensor raw 6 ch, Ts=128 | LACA query-dependent lag (mu_i, sigma_i), 4 heads | Mamba 2 x (d_state 16, d_conv 2) | loss CE + 0.1 InfoNCE + 0.1 KL | 20 epochs, batch 32, AdamW 0.0001


In [2]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

Installing mamba_ssm + causal-conv1d (python=cp312, torch=2.10, cuda=12.8, cxx11abi=True)
    pip install ['https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl', '--no-deps'] failed (exit 1): ERROR: causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl is not a supported wheel on this platform.

  no complete prebuilt-wheel match — trying a time-capped source build
mamba_ssm install: source build
✅ FUSION BACKBONE: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 3. Data loading & preprocessing
Mission-level split, strongest-64 bin selection from training audio only, 2.56 s windows with real timestamps, and sensor standardization with training-split statistics. Leakage and timestamp checks are included.

In [3]:
# ==============================================================================
# DATA — TII UAV Realistic Fault Dataset: parse missions -> mission-level split
# -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows with
# real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
split_windows = {}
for split in ("train", "val", "test"):
    split_windows[split] = []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                split_windows[split].extend(slice_windows(parsed_missions[m], label, SELECTED_BINS))
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps reach HTT/LACA ✅")
del _tau_a, _tau_s

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git ...


Cloning into '/kaggle/working/tii_uav_dataset'...


Class 0 (19 flights) -> train 13, val 2, test 4
Class 1 (20 flights) -> train 14, val 3, test 3
Class 2 (20 flights) -> train 14, val 3, test 3
Class 3 (20 flights) -> train 14, val 3, test 3
Class 4 (20 flights) -> train 14, val 3, test 3


Updating files: 100% (217/217), done.


parse train:   0%|          | 0/69 [00:00<?, ?it/s]

parse val:   0%|          | 0/14 [00:00<?, ?it/s]

parse test:   0%|          | 0/16 [00:00<?, ?it/s]

[LEAKAGE CHECK] strongest-64 bins computed from 69 TRAINING missions only (0 of 30 val/test missions) -> frozen for all splits: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
train: 4934 windows | audio 64x64, sensor 128x6
val  : 954 windows | audio 64x64, sensor 128x6
test : 1130 windows | audio 64x64, sensor 128x6
[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | train mean [0.05299999937415123, -0.0430000014603138, -9.78600025177002, 0.004000000189989805, 0.003000000026077032, -0.0] vs val's own mean [0.029999999329447746, -0.061000000685453415, -9.748000144958496, 0.004999999888241291, 0.003000000026077032, -0.0] (not used)
[TIMESTAMP CHECK] 4934 training windows: 4934 distinct 

## 4. Model definition — query-dependent lag-adaptive LACA
**What v18's `PairwiseLACA` did.** It takes `H_m` (B, T_m, 256) as queries and `H_n` (B, T_n, 256) as keys and values, plus `tau_m` (B, T_m) and `tau_n` (B, T_n), both in normalized window time.
- Q, K and V are reshaped to (B, 4, T, 64), and `scores = QKᵀ/√64` has shape (B, 4, T_m, T_n).
- `delta_tau = tau_m[:, :, None] − tau_n[:, None, :]`, i.e. query minus key.
- `laca_as` uses audio queries (T_m = 64) against sensor keys (T_n = 128); `laca_sa` is the reverse.

**v21.** `lag_predictor = Linear(256, 2)` reads the same `H_m` that feeds W_q. It produces μ and σ of shape (B, T_m), shared across the 4 heads. The bias `−½((Δτ − μ)/σ)²` of shape (B, 1, T_m, T_n) is added to the content scores. After that addition, and before the softmax, **every forward pass checks that the scores are finite**. If not, it prints the sample, head, query and key involved, with μ, σ, Δτ, the lag bias and the content score, and raises.

`lag_mode="none"` is content-only attention, which is v18 at β = 0. It is used only to load the control's checkpoints in Part B.

The cell ends by tracing the real tensor shapes on 2 training windows.

In [4]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention with a QUERY-DEPENDENT lag bias:
#    -0.5 ((Δτ_ij - mu_i) / sigma_i)^2, mu_i / sigma_i predicted from each query token; 4 heads)
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding + time embedding(tau)); modality 0 = audio, 1 = sensor."""

    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        return self.norm(z + e_m + self.time_embedding(tau))


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m (queries, H_m / tau_m) to n (keys + values, H_n / tau_n).

    lag_mode "query" (v21): each query token i predicts its own preferred lag mu_i and tolerance sigma_i from H_m[i]
    (the same representation that feeds W_q) and adds  bias_ij = -0.5 * ((Δτ_ij - mu_i) / sigma_i)^2  to the
    content scores, Δτ_ij = tau_m[i] - tau_n[j]. No beta, no global g(Δτ); nothing label-dependent.
    lag_mode "none": content-only attention (= v18 at beta = 0); used only to load the reused control's weights."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, lag_mode: str = "query"):
        super().__init__()
        assert lag_mode in ("query", "none"), lag_mode
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.lag_mode = lag_mode
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        # RNG alignment: v18 built its global g(Δτ) = Linear(1,32) -> Tanh -> Linear(32,1) here. The same draw is made
        # and discarded (it is not part of the model), so every later layer's initial weights and the training RNG
        # stream stay identical to the reused v18 control — the same idea as RNG_REFERENCE_STACK below.
        nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        if lag_mode == "query":
            # Small init from a FORKED RNG: mu ~ 0 and sigma ~ softplus(0) + MIN_SIGMA at start (no tanh saturation),
            # and the global RNG stream is left exactly where the control's was.
            with torch.random.fork_rng(devices=[]):
                self.lag_predictor = nn.Linear(d_model, 2)
                nn.init.normal_(self.lag_predictor.weight, mean=0.0, std=LAG_PREDICTOR_INIT_STD)
                nn.init.zeros_(self.lag_predictor.bias)
        self.capture = None        # list -> (mu, sigma) of every forward pass is appended (diagnostics)
        self.finite_checks = 0     # number of attention-score tensors checked for NaN / Inf

    def lag_params(self, H_m):
        """(mu, sigma), each (B, T_m): one preferred lag and one tolerance per query token."""
        raw = self.lag_predictor(H_m)                        # (B, T_m, 2)
        mu = torch.tanh(raw[..., 0]) * MAX_LAG
        sigma = F.softplus(raw[..., 1]) + MIN_SIGMA
        return mu, sigma

    def _assert_finite(self, scores, content, mu, sigma, delta_tau, lag_bias):
        self.finite_checks += 1
        bad = ~torch.isfinite(scores)
        if not bool(bad.any()):
            return
        b, h, i, j = bad.nonzero()[0].tolist()
        print("🚨" * 45)
        print(f"🚨  NON-FINITE ATTENTION SCORES in {self.lag_mode} LACA (forward #{self.finite_checks}): "
              f"{int(bad.sum())} of {bad.numel()} entries; first at sample {b}, head {h}, query {i}, key {j}")
        print(f"🚨  mu[{b},{i}] = {float(mu[b, i])!r} | sigma[{b},{i}] = {float(sigma[b, i])!r} | Δτ[{b},{i},{j}] = "
              f"{float(delta_tau[b, i, j])!r} | lag_bias = {float(lag_bias[b, i, j])!r} | content score = "
              f"{float(content[b, h, i, j])!r}")
        print(f"🚨  batch: sigma min {float(sigma.min())!r} | mu range [{float(mu.min())!r}, {float(mu.max())!r}] | "
              f"max |Δτ| {float(delta_tau.abs().max())!r} | non-finite mu/sigma: {int((~torch.isfinite(mu)).sum())}/"
              f"{int((~torch.isfinite(sigma)).sum())} | non-finite content scores: {int((~torch.isfinite(content)).sum())}")
        print("🚨" * 45)
        raise FloatingPointError("non-finite attention scores in query-dependent LACA — diagnostics printed above")

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale             # (B, heads, T_m, T_n) content scores
        if self.lag_mode == "query":
            mu, sigma = self.lag_params(H_m)                                     # (B, T_m) each
            delta_tau = tau_m.unsqueeze(2) - tau_n.unsqueeze(1)                  # (B, T_m, T_n), query minus key
            lag_bias = -0.5 * ((delta_tau - mu.unsqueeze(-1)) / sigma.unsqueeze(-1)) ** 2   # (B, T_m, T_n)
            content = scores
            scores = content + lag_bias.unsqueeze(1)                             # shared by the heads
            self._assert_finite(scores, content, mu, sigma, delta_tau, lag_bias)
            if self.capture is not None:
                self.capture.append((mu.detach().float().cpu(), sigma.detach().float().cpu()))

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, lag_mode: str = "query"):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, lag_mode=lag_mode)   # audio queries -> sensor keys / values
        self.laca_sa = PairwiseLACA(d_model, lag_mode=lag_mode)   # sensor queries -> audio keys / values
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, lag_mode: str = "query"):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.lag_mode = lag_mode

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model, lag_mode=lag_mode)
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts, ts_lag=None):
        """ts_lag (Part B only): sensor timestamps seen by the LACA lag term alone; HTT keeps using ts."""
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts if ts_lag is None else ts_lag)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules (v21: query-dependent lag, no beta, no g)."""
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert not hasattr(laca, "beta") and not hasattr(laca, "lag_bias"), f"{name}: global lag bias still present"
        if model.lag_mode == "query":
            lp = laca.lag_predictor
            assert (lp.in_features, lp.out_features) == (model.d_model, 2)
    assert not any("beta" in n for n, _ in model.named_parameters()), "a beta parameter exists"
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe = AeroSyncMamba()
_lp = sum(p.numel() for n, p in _probe.named_parameters() if "lag_predictor" in n)
print(f"Model: {sum(p.numel() for p in _probe.parameters()):,} params (lag predictors {_lp} = 2 x Linear(256, 2); v18's "
      f"2 x 97 g(Δτ) params removed) | fusion blocks: {type(_probe.mamba_blocks[0]).__name__} x "
      f"{len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} params) | backbone: {FUSION_BACKBONE}")

# --- real tensor shapes through both LACA directions (2 training windows, CPU, eval, no RNG) ---
LAG_PROBE_IDX = np.linspace(0, len(train_ds) - 1, BATCH_SIZE).astype(int)   # spread over the training set
_items = [train_ds[int(i)] for i in LAG_PROBE_IDX[:2]]
_b = {k: torch.stack([it[k] for it in _items]) for k in ("x_a", "x_s", "tau_a", "tau_s")}
_probe.eval()
with torch.no_grad():
    _Ha = _probe.htt(_probe.audio_encoder(_b["x_a"]), _b["tau_a"], modality_idx=0)
    _Hs = _probe.htt(_probe.sensor_encoder(_b["x_s"].transpose(1, 2)).transpose(1, 2), _b["tau_s"], modality_idx=1)
    for _name, _laca, _Hm, _Hn, _tm, _tn in (("laca_as", _probe.laca.laca_as, _Ha, _Hs, _b["tau_a"], _b["tau_s"]),
                                             ("laca_sa", _probe.laca.laca_sa, _Hs, _Ha, _b["tau_s"], _b["tau_a"])):
        _B, _Tm, _ = _Hm.shape
        _Tn = _Hn.shape[1]
        _Q = _laca.w_q(_Hm).view(_B, _Tm, _laca.num_heads, _laca.head_dim).transpose(1, 2)
        _K = _laca.w_k(_Hn).view(_B, _Tn, _laca.num_heads, _laca.head_dim).transpose(1, 2)
        _sc = torch.matmul(_Q, _K.transpose(-2, -1)) * _laca.scale
        _mu, _sigma = _laca.lag_params(_Hm)
        _dt = _tm.unsqueeze(2) - _tn.unsqueeze(1)
        _lb = -0.5 * ((_dt - _mu.unsqueeze(-1)) / _sigma.unsqueeze(-1)) ** 2
        print(f"[SHAPES] {_name}: H_m {tuple(_Hm.shape)} -> Q {tuple(_Q.shape)}, K {tuple(_K.shape)} -> content scores "
              f"{tuple(_sc.shape)} | tau_m {tuple(_tm.shape)}, tau_n {tuple(_tn.shape)} -> Δτ = tau_m - tau_n "
              f"{tuple(_dt.shape)} (range {float(_dt.min()):+.3f}..{float(_dt.max()):+.3f}) | mu, sigma {tuple(_mu.shape)} "
              f"(from H_m) -> lag bias {tuple(_lb.unsqueeze(1).shape)} broadcast over {_laca.num_heads} heads "
              f"(init range {float(_lb.min()):.3f}..{float(_lb.max()):.3f})")
del _probe, _items, _b, _Ha, _Hs, _Q, _K, _sc, _mu, _sigma, _dt, _lb

Model: 1,689,622 params (lag predictors 1028 = 2 x Linear(256, 2); v18's 2 x 97 g(Δτ) params removed) | fusion blocks: MambaBlock x 2 (874,496 params) | backbone: mamba_ssm.Mamba (real selective-scan SSM, CUDA)
[SHAPES] laca_as: H_m (2, 64, 256) -> Q (2, 4, 64, 64), K (2, 4, 128, 64) -> content scores (2, 4, 64, 128) | tau_m (2, 64), tau_n (2, 128) -> Δτ = tau_m - tau_n (2, 64, 128) (range -0.999..+0.996) | mu, sigma (2, 64) (from H_m) -> lag bias (2, 1, 64, 128) broadcast over 4 heads (init range -0.982..-0.000)
[SHAPES] laca_sa: H_m (2, 128, 256) -> Q (2, 4, 128, 64), K (2, 4, 64, 64) -> content scores (2, 4, 128, 64) | tau_m (2, 128), tau_n (2, 64) -> Δτ = tau_m - tau_n (2, 128, 64) (range -0.996..+0.999) | mu, sigma (2, 128) (from H_m) -> lag bias (2, 1, 128, 64) broadcast over 4 heads (init range -0.956..-0.000)


## 5. Metrics, μ/σ diagnostics, and the training function
**Evaluation.** `evaluate_probs` returns:
- the 4 standard metrics;
- the full 5×5 confusion matrix and per-class recall, including **Class 3 and Class 4 recall**;
- the **3→4** count (true 3, predicted 4) and the **4→3** count;
- adjacent and distant errors.

**μ/σ statistics.** `lag_param_stats` records μ and σ from every forward pass it covers and summarizes them per direction: mean, std, min and max; mean |μ| and its value in ms; the share of μ near tanh's ±0.5 limit; and the share of σ near the 0.02 floor.

**`train_one_run(seed)`:**
1. Resets every RNG and builds a fresh model.
2. Checks that the torch RNG state after construction matches the control's for that seed.
3. Measures **μ/σ on a real batch before any training step** and stops if they are saturated or extreme.
4. Trains with the unchanged class-weighted CE loss, printing μ/σ on the probe batch every epoch.
5. Measures **μ/σ over every test-set query token at the final epoch**.
6. Loads the best checkpoint and evaluates it. It also reports how many attention-score tensors passed the finite check.

In [5]:
# ==============================================================================
# TRAINING — loss (unchanged class-weighted CE + InfoNCE + KL), metrics
# (4 standard + confusion matrix + Class 3/4 recall + 3->4 / 4->3 counts),
# mu/sigma diagnostics, and train_one_run(seed): one query-dependent-LACA model
# end-to-end (20 epochs, best-validation checkpoint), evaluated on the test set.
# ==============================================================================
class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
CLASS_WEIGHTS = train_class_weights(split_windows["train"]).to(device)
TEST_LABELS = np.array([w["label"] for w in split_windows["test"]], dtype=np.int64)   # test-window order, fixed


from sklearn.metrics import confusion_matrix


def evaluate_probs(y_true, y_prob):
    """4 standard metrics + full 5x5 confusion matrix + per-class recall + the Class 3/4 confusions + adjacent / distant."""
    cm = confusion_matrix(y_true, y_prob.argmax(axis=1), labels=list(range(N_CLASSES)))
    recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
    dist = np.abs(np.arange(N_CLASSES)[:, None] - np.arange(N_CLASSES)[None, :])
    return {**compute_metrics(y_true, y_prob), "confusion_matrix": cm,
            **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
            "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
            "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3]),
            "adjacent_errors": int(cm[dist == 1].sum()), "distant_errors": int(cm[dist >= 2].sum())}


def print_confusion(ev, tag):
    cm = ev["confusion_matrix"]
    print(f"[CONFUSION {tag}] rows = TRUE severity, cols = PREDICTED severity (test set, {int(cm.sum())} windows)")
    print("            " + "".join(f"pred {j:>2} " for j in range(N_CLASSES)) + "|  recall")
    for i in range(N_CLASSES):
        print(f"   true {i:>2} " + "".join(f"{int(cm[i, j]):>8}" for j in range(N_CLASSES)) + f"  |  {ev[f'recall_c{i}']:.4f}")
    print(f"[CLASS 3/4 {tag}] Class 3 recall {ev['class3_recall']:.4f} | Class 4 recall {ev['class4_recall']:.4f} | "
          f"3->4 (true 3, pred 4) {ev['err_3to4']} | 4->3 (true 4, pred 3) {ev['err_4to3']} | adjacent errors "
          f"{ev['adjacent_errors']} | distant errors {ev['distant_errors']}")


# --- mu / sigma diagnostics ---
LACA_DIRECTIONS = (("as", "laca_as", "audio queries -> sensor keys"), ("sa", "laca_sa", "sensor queries -> audio keys"))


def set_capture(model, on):
    for _, attr, _ in LACA_DIRECTIONS:
        getattr(model.laca, attr).capture = [] if on else None


def collect_capture(model):
    """Summarize every captured (mu, sigma) over all query tokens, per direction; switches capture off."""
    out = {}
    for key, attr, _ in LACA_DIRECTIONS:
        cap = getattr(model.laca, attr).capture
        mu = torch.cat([m.reshape(-1) for m, _ in cap])
        sigma = torch.cat([s.reshape(-1) for _, s in cap])
        out[key] = {"n_tokens": int(mu.numel()),
                    "mu_mean": float(mu.mean()), "mu_std": float(mu.std()), "mu_min": float(mu.min()),
                    "mu_max": float(mu.max()), "mu_absmean": float(mu.abs().mean()),
                    "mu_sat_frac": float((mu.abs() > INIT_MU_SATURATION).float().mean()),
                    "sigma_mean": float(sigma.mean()), "sigma_std": float(sigma.std()), "sigma_min": float(sigma.min()),
                    "sigma_max": float(sigma.max()), "sigma_floor_frac": float((sigma < MIN_SIGMA + 0.01).float().mean())}
    set_capture(model, False)
    return out


def print_lag_stats(stats, tag, when):
    for key, _, label in LACA_DIRECTIONS:
        s = stats[key]
        print(f"{tag} [MU/SIGMA {when}] {label} ({s['n_tokens']:,} query tokens): mu mean {s['mu_mean']:+.4f} std "
              f"{s['mu_std']:.4f} min {s['mu_min']:+.4f} max {s['mu_max']:+.4f} (mean |mu| {s['mu_absmean']:.4f} = "
              f"{s['mu_absmean'] * WINDOW_SEC * 1000:.0f} ms; |mu| > {INIT_MU_SATURATION:.2f}: {s['mu_sat_frac']:.2%}) | "
              f"sigma mean {s['sigma_mean']:.4f} std {s['sigma_std']:.4f} min {s['sigma_min']:.4f} max "
              f"{s['sigma_max']:.4f} (within 0.01 of the {MIN_SIGMA} floor: {s['sigma_floor_frac']:.2%})")


def lag_param_stats_probe(model):
    """mu / sigma on the fixed real probe batch (LAG_PROBE_IDX training windows), eval mode, RNG untouched."""
    items = [train_ds[int(i)] for i in LAG_PROBE_IDX]
    xa, xs, ta, ts = (torch.stack([it[k] for it in items]).to(device) for k in ("x_a", "x_s", "tau_a", "tau_s"))
    was_training = model.training
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=cuda_devices), torch.no_grad():
        model.eval()
        set_capture(model, True)
        model(xa, xs, ta, ts)
        stats = collect_capture(model)
    model.train(was_training)
    return stats


def check_init_stats(stats):
    """Problems with the initial mu / sigma (saturated mu or extreme sigma); empty list = fine."""
    problems = []
    for key, _, label in LACA_DIRECTIONS:
        s = stats[key]
        if s["mu_absmean"] > INIT_MU_ABSMEAN_MAX:
            problems.append(f"{label}: mean |mu| {s['mu_absmean']:.4f} > {INIT_MU_ABSMEAN_MAX} (a seed-specific starting "
                            f"lag of {s['mu_mean'] * WINDOW_SEC * 1000:+.0f} ms; not saturation)")
        if max(abs(s["mu_min"]), abs(s["mu_max"])) > INIT_MU_SATURATION:
            problems.append(f"{label}: |mu| reaches {max(abs(s['mu_min']), abs(s['mu_max'])):.4f} > {INIT_MU_SATURATION} (tanh saturation)")
        if not (INIT_SIGMA_RANGE[0] <= s["sigma_min"] and s["sigma_max"] <= INIT_SIGMA_RANGE[1]):
            problems.append(f"{label}: sigma range [{s['sigma_min']:.4f}, {s['sigma_max']:.4f}] outside {INIT_SIGMA_RANGE}")
    return problems


def predict_probs_tau(model, loader, mode="correct", capture=False):
    """Softmax probabilities over a loader. mode: 'correct' (true tau_s), 'all' (shuffled tau_s everywhere: HTT + LACA),
    'laca_only' (HTT sees true tau_s, the LACA lag term sees shuffled tau_s). Optionally captures mu / sigma."""
    model.eval()
    if capture:
        set_capture(model, True)
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            if mode == "correct":
                logits = model(xa, xs, ta, ts)[0]
            else:
                ts_shuf = batch["tau_s_shuffled"].to(device)
                logits = model(xa, xs, ta, ts_shuf)[0] if mode == "all" else model(xa, xs, ta, ts, ts_lag=ts_shuf)[0]
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    stats = collect_capture(model) if capture else None
    return np.concatenate(probs), np.concatenate(labels), stats


def finite_check_count(model):
    return sum(getattr(model.laca, attr).finite_checks for _, attr, _ in LACA_DIRECTIONS)


def flat_stats(stats, prefix):
    return {f"{prefix}_{key}_{k}": v for key, s in stats.items() for k, v in s.items() if k != "n_tokens"}


def train_one_run(seed: int):
    """Train one query-dependent-LACA model with `seed` applied to every RNG, evaluate it on the test set.
    Returns (trained model on CPU, result dict incl. test probabilities, confusion matrix and mu / sigma stats)."""
    t0 = time.time()
    tag = f"[query-lag seed={seed}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(lag_mode="query").to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model)
    ctrl = V18_BETA0_CONTROL[seed]
    rng_ok = rng_sha1[:12] == ctrl["rng_sha1"]
    print(f"{tag} fusion {type(model.mamba_blocks[0]).__name__} ({mamba_params:,} params) | [PAIRING] torch RNG after "
          f"construction {rng_sha1[:12]} vs control {ctrl['rng_sha1']} -> "
          + ("✅ identical: every shared layer starts from the control's weights, same shuffle / dropout stream" if rng_ok
             else "🚨 MISMATCH — construction consumed the RNG differently from the control; pairing broken"))

    # --- mu / sigma at initialization, on a real batch, BEFORE any training step ---
    init_stats = lag_param_stats_probe(model)
    print_lag_stats(init_stats, tag, "INIT, before any training step")
    problems = check_init_stats(init_stats)
    if problems:
        print("🚨" * 45)
        for p in problems:
            print(f"🚨  BAD LAG-PREDICTOR INIT: {p}")
        print(f"🚨  -> a different init scale is needed (LAG_PREDICTOR_INIT_STD = {LAG_PREDICTOR_INIT_STD} now).")
        print("🚨" * 45)
        if HALT_ON_BAD_LAG_INIT:
            raise RuntimeError("lag-predictor initialization is saturated / extreme — see the diagnostics above")
    else:
        print(f"{tag} [INIT CHECK] ✅ mu starts near 0 and is not saturated (mean |mu| <= {INIT_MU_ABSMEAN_MAX}, every |mu| < "
              f"{INIT_MU_SATURATION}) and sigma moderate (all within {INIT_SIGMA_RANGE}; softplus(0) + {MIN_SIGMA} = "
              f"{math.log(2) + MIN_SIGMA:.3f})")

    criterion = AeroSyncLoss(CLASS_WEIGHTS)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        for batch in tqdm(train_loader, desc=f"query-lag seed {seed} epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        ep = lag_param_stats_probe(model)
        val_probs, val_labels = predict_probs(model, val_loader)
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        n = len(train_loader)
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | mu as {ep['as']['mu_mean']:+.3f}±{ep['as']['mu_std']:.3f} sa "
              f"{ep['sa']['mu_mean']:+.3f}±{ep['sa']['mu_std']:.3f} | sigma as {ep['as']['sigma_mean']:.3f} sa "
              f"{ep['sa']['sigma_mean']:.3f} | train loss {running['loss'] / n:.4f} (cls {running['loss_cls'] / n:.4f}, "
              f"align {running['loss_align'] / n:.4f}, cons {running['loss_cons'] / n:.4f}) | val macro-F1 {val_f1:.4f}"
              f"{'  * best' if improved else ''}")

    # --- mu / sigma at the FINAL epoch over every test-set query token ---
    _, _, final_stats = predict_probs_tau(model, test_loader, "correct", capture=True)
    print_lag_stats(final_stats, tag, f"FINAL EPOCH {EPOCHS}, test set")

    model.load_state_dict(best_state)
    test_probs, test_labels, best_stats = predict_probs_tau(model, test_loader, "correct", capture=True)
    print_lag_stats(best_stats, tag, f"BEST CHECKPOINT (epoch {best_epoch}), test set")
    ev = evaluate_probs(test_labels, test_probs)
    n_checks = finite_check_count(model)
    print(f"{tag} [NaN/Inf CHECK] {n_checks:,} attention-score tensors checked (every forward pass, both directions, "
          f"training + evaluation): none non-finite ✅ (the assertion never triggered)")
    run_name = f"query_lag_seed{seed}"
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "seed": seed,
        **ev,
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "n_test_windows": len(test_labels),
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "paired_with_control": rng_ok,
        "finite_checks": n_checks,
        "init_stats": init_stats, "final_stats": final_stats, "best_stats": best_stats,
        **flat_stats(init_stats, "init"), **flat_stats(final_stats, "final"),
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model.cpu(), result

## 6. Part A — control reuse check, then 5 runs
**The control is reused, not retrained.** For each seed, the confusion matrix embedded in Section 2 must reproduce v18's accuracy, macro-F1 and balanced accuracy exactly. That confirms the confusion matrices belong to the v18 runs.

**The 5 runs.** After each run the driver:
- prints the full confusion matrix;
- prints the paired differences against the control;
- appends a row to `/kaggle/working/query_lag_laca_results.csv`, including the initial and final-epoch μ/σ statistics;
- frees the GPU and prints progress.

In [6]:
# ==============================================================================
# PART A DRIVER — verify the reused control, then one query-dependent-LACA run
# per seed: confusion matrix, paired diffs vs control, CSV row, GPU freed.
# ==============================================================================
import pandas as pd

METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]

# ---------------------------------------------------------------- the REUSED control (no training)
print("=" * 100)
print("CONTROL GROUP — v18's beta=0.0 runs, REUSED (not retrained): the current LACA with its global lag bias disabled.")
print("=" * 100)
control = {}
CONTROL_MATCH = {}
for seed in SEEDS:
    ref = V18_BETA0_CONTROL[seed]
    cm = np.array(ref["confusion_matrix"])
    y_true = np.repeat(np.arange(N_CLASSES), cm.sum(axis=1))
    y_pred = np.concatenate([np.repeat(np.arange(N_CLASSES), cm[i]) for i in range(N_CLASSES)])
    assert np.array_equal(np.bincount(y_true, minlength=N_CLASSES), np.bincount(TEST_LABELS, minlength=N_CLASSES)), \
        "control confusion matrix does not cover this test set"
    from_cm = {"accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro"),
               "balanced_accuracy": balanced_accuracy_score(y_true, y_pred)}
    CONTROL_MATCH[seed] = max(abs(from_cm[m] - ref[m]) for m in from_cm)
    recall = cm.diagonal() / cm.sum(axis=1)
    dist = np.abs(np.arange(N_CLASSES)[:, None] - np.arange(N_CLASSES)[None, :])
    control[seed] = {**{m: ref[m] for m in METRICS}, "confusion_matrix": cm,
                     **{f"recall_c{c}": float(recall[c]) for c in range(N_CLASSES)},
                     "class3_recall": float(recall[3]), "class4_recall": float(recall[4]),
                     "err_3to4": int(cm[3, 4]), "err_4to3": int(cm[4, 3]),
                     "adjacent_errors": int(cm[dist == 1].sum()), "distant_errors": int(cm[dist >= 2].sum())}
    print(f"[CONTROL seed={seed}] REUSED v18 run: accuracy {ref['accuracy']:.4f} | macro-F1 {ref['macro_f1']:.4f} | "
          f"balanced acc {ref['balanced_accuracy']:.4f} | AUROC {ref['macro_auroc']:.4f} | metrics recomputed from its "
          f"confusion matrix match v18 to {CONTROL_MATCH[seed]:.1e} "
          + ("✅ EXACT" if CONTROL_MATCH[seed] < 1e-12 else "❌ MISMATCH"))
    print_confusion(control[seed], f"control seed={seed}")
assert all(v < 1e-12 for v in CONTROL_MATCH.values()), "embedded control confusion matrices do not reproduce v18"

# ---------------------------------------------------------------- 5 new runs
RESULTS_CSV_FIELDS = None
if RESULTS_CSV.exists():
    RESULTS_CSV.rename(RESULTS_CSV.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))

run_results = {}
t_driver = time.time()
for i, seed in enumerate(SEEDS, start=1):
    print("\n" + "=" * 100 + f"\nPART A RUN {i}/{len(SEEDS)} — query-dependent lag LACA, seed={seed}\n" + "=" * 100)
    model, result = train_one_run(seed)
    run_results[seed] = result
    c = control[seed]
    print(f"[query-lag seed={seed}] TEST: accuracy {result['accuracy']:.4f} | macro-F1 {result['macro_f1']:.4f} | "
          f"balanced accuracy {result['balanced_accuracy']:.4f} | macro-AUROC {result['macro_auroc']:.4f} | best epoch "
          f"{result['best_epoch']} | {result['train_seconds']:.0f}s")
    print_confusion(result, f"query-lag seed={seed}")
    print(f"[PAIR seed={seed}] vs REUSED control: macro-F1 {result['macro_f1'] - c['macro_f1']:+.4f} | Class 4 recall "
          f"{result['class4_recall'] - c['class4_recall']:+.4f} | Class 3 recall {result['class3_recall'] - c['class3_recall']:+.4f} "
          f"| 3->4 {result['err_3to4'] - c['err_3to4']:+d} | 4->3 {result['err_4to3'] - c['err_4to3']:+d}")

    row = {k: v for k, v in result.items() if not isinstance(v, (dict, np.ndarray))}
    row.update(run=i, confusion_matrix=json.dumps(result["confusion_matrix"].tolist()),
               control_macro_f1=c["macro_f1"], control_class4_recall=c["class4_recall"],
               finished_at=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    if RESULTS_CSV_FIELDS is None:
        RESULTS_CSV_FIELDS = ["run"] + [k for k in row if k != "run"]
    write_header = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elapsed = time.time() - t_driver
    print(f"\n>>> Part A {i}/{len(SEEDS)} complete: seed={seed} finished | elapsed {elapsed / 60:.1f} min | "
          f"est. remaining {elapsed / i * (len(SEEDS) - i) / 60:.1f} min")

print(f"\nPart A: all {len(SEEDS)} runs finished in {(time.time() - t_driver) / 60:.1f} min. Results: {RESULTS_CSV}")

CONTROL GROUP — v18's beta=0.0 runs, REUSED (not retrained): the current LACA with its global lag bias disabled.
[CONTROL seed=42] REUSED v18 run: accuracy 0.9088 | macro-F1 0.9057 | balanced acc 0.9030 | AUROC 0.9892 | metrics recomputed from its confusion matrix match v18 to 0.0e+00 ✅ EXACT
[CONFUSION control seed=42] rows = TRUE severity, cols = PREDICTED severity (test set, 1130 windows)
            pred  0 pred  1 pred  2 pred  3 pred  4 |  recall
   true  0      283       0       1       0       2  |  0.9895
   true  1        3     197       2       1       2  |  0.9610
   true  2        0       1     172      35       2  |  0.8190
   true  3        0       0       8     204      16  |  0.8947
   true  4        0       0       2      28     171  |  0.8507
[CLASS 3/4 control seed=42] Class 3 recall 0.8947 | Class 4 recall 0.8507 | 3->4 (true 3, pred 4) 16 | 4->3 (true 4, pred 3) 28 | adjacent errors 93 | distant errors 10
[CONTROL seed=123] REUSED v18 run: accuracy 0.8876 | macro-

query-lag seed 42 epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

[query-lag seed=42] epoch 01/20 | mu as +0.228±0.007 sa -0.293±0.041 | sigma as 0.689 sa 0.540 | train loss 1.4173 (cls 1.0706, align 3.1022, cons 0.3651) | val macro-F1 0.8314  * best


query-lag seed 42 epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

[query-lag seed=42] epoch 02/20 | mu as +0.273±0.013 sa -0.250±0.059 | sigma as 0.816 sa 0.586 | train loss 0.9244 (cls 0.5962, align 2.6646, cons 0.6176) | val macro-F1 0.6799


...

[query-lag seed=99] epoch 18/20 | mu as +0.320±0.033 sa -0.007±0.013 | sigma as 0.600 sa 0.631 | train loss 0.3188 (cls 0.1480, align 1.2744, cons 0.4340) | val macro-F1 0.8435


query-lag seed 99 epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

[query-lag seed=99] epoch 19/20 | mu as +0.322±0.033 sa -0.000±0.013 | sigma as 0.593 sa 0.612 | train loss 0.2879 (cls 0.1168, align 1.2731, cons 0.4379) | val macro-F1 0.8824


query-lag seed 99 epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

[query-lag seed=99] epoch 20/20 | mu as +0.322±0.033 sa -0.004±0.013 | sigma as 0.594 sa 0.603 | train loss 0.2894 (cls 0.1195, align 1.2627, cons 0.4368) | val macro-F1 0.8837
[query-lag seed=99] [MU/SIGMA FINAL EPOCH 20, test set] audio queries -> sensor keys (72,320 query tokens): mu mean +0.3164 std 0.0336 min +0.1999 max +0.3918 (mean |mu| 0.3164 = 810 ms; |mu| > 0.45: 0.00%) | sigma mean 0.5796 std 0.1646 min 0.3047 max 1.0178 (within 0.01 of the 0.02 floor: 0.00%)
[query-lag seed=99] [MU/SIGMA FINAL EPOCH 20, test set] sensor queries -> audio keys (144,640 query tokens): mu mean -0.0039 std 0.0135 min -0.0462 max +0.0564 (mean |mu| 0.0118 = 30 ms; |mu| > 0.45: 0.00%) | sigma mean 0.6150 std 0.2554 min 0.1322 max 1.0807 (within 0.01 of the 0.02 floor: 0.00%)
[query-lag seed=99] [MU/SIGMA BEST CHECKPOINT (epoch 16), test set] audio queries -> sensor keys (72,320 query tokens): mu mean +0.3256 std 0.0338 min +0.2062 max +0.3983 (mean |mu| 0.3256 = 833 ms; |mu| > 0.45: 0.00%) | sigm

## 7. Part A — paired statistics and the learned μ/σ
**Paired statistics.** The same statistics as every earlier β experiment, computed separately for **macro-F1** and **Class 4 recall** (new architecture minus the reused control): per-seed differences, mean paired difference, std of the paired differences, win count, a **paired t-test (primary)** and a Wilcoxon signed-rank test (secondary).

**Why the t-test is primary.** With 5 pairs, the exact two-sided Wilcoxon p-value can't go below 2/2⁵ = 0.0625, so it can never reach 0.05. Class 4 recall also moves in steps of 1/201, which makes ties likely. The t-test assumes the differences are roughly normal, which 5 pairs can't check, so the Wilcoxon test is reported alongside it.

**Also reported per seed:** Class 3 recall, the 3→4 and 4→3 counts, and the learned μ/σ for each direction.

**Where μ ended up.** Mean |μ| below 0.02 (about 50 ms) counts as "no preferred lag learned". A μ spread across tokens below 0.01 counts as "one shared lag", i.e. no query dependence. The largest mean |μ| is divided by the smallest across seeds; above 3, the result depends on the seed.

In [10]:
# ==============================================================================
# PART A ANALYSIS — paired stats (macro-F1, Class 4 recall), Class 3/4 detail,
# learned mu / sigma per seed, verdicts.
# ==============================================================================
from scipy import stats

dfA = pd.read_csv(RESULTS_CSV)
done = [s for s in SEEDS if s in set(dfA["seed"])]
n = len(done)
rowA = {s: dfA[dfA["seed"] == s].iloc[-1] for s in done}
print(f"Part A runs finished: {n}/{len(SEEDS)} -> seeds {done} (each paired with its REUSED v18 beta=0.0 control)")
assert n >= 2, "need at least 2 finished seeds for any paired statistic"


def paired_stats(new, base):
    d = np.asarray(new, dtype=float) - np.asarray(base, dtype=float)
    k = len(d)
    out = {"d": d, "n": k, "mean": float(d.mean()), "sd": float(d.std(ddof=1)), "wins": int((d > 0).sum()),
           "losses": int((d < 0).sum()), "ties": int((d == 0).sum())}
    out["p_t"] = float(stats.ttest_rel(new, base).pvalue) if out["sd"] > 0 else float("nan")
    try:
        out["p_w"] = float(stats.wilcoxon(d, alternative="two-sided", method="exact").pvalue)
    except Exception:
        try:
            out["p_w"] = float(stats.wilcoxon(d, alternative="two-sided").pvalue)
        except Exception:
            out["p_w"] = float("nan")
    return out


def verdict(P, what):
    if P["mean"] > 0 and P["wins"] >= P["n"] - 1 and (P["mean"] > P["sd"] or P["p_t"] < PAIRED_ALPHA):
        return "WELL-SUPPORTED improvement"
    if P["mean"] < 0 and P["losses"] >= P["n"] - 1 and (-P["mean"] > P["sd"] or P["p_t"] < PAIRED_ALPHA):
        return "REVERSED (control better)"
    return "NOT WELL-SUPPORTED"


PA = {}
for q, label in (("macro_f1", "macro-F1"), ("class4_recall", "Class 4 recall")):
    base = np.array([control[s][q] for s in done])
    new = np.array([float(rowA[s][q]) for s in done])
    P = paired_stats(new, base)
    PA[q] = P
    print("\n" + "=" * 100)
    print(f"PAIRED — {label}: query-dependent lag LACA minus REUSED v18 beta=0.0 control, {n} matched seeds")
    print("=" * 100)
    print(f"  {'seed':>6} {'control':>9} {'query-lag':>10} {'diff':>9}")
    for s, b0, b1, x in zip(done, base, new, P["d"]):
        print(f"  {s:>6} {b0:>9.4f} {b1:>10.4f} {x:>+9.4f}")
    print(f"  mean paired difference {P['mean']:+.4f} | std of paired differences {P['sd']:.4f} | query-lag better on "
          f"{P['wins']}/{n} (worse {P['losses']}, ties {P['ties']})")
    print(f"  paired t-test (PRIMARY) two-sided p = {P['p_t']:.4f} | Wilcoxon signed-rank p = {P['p_w']:.4f} "
          f"(smallest achievable with n={n}: {2 / 2 ** n:.4f})")
    print(f"  VERDICT ({label}): {verdict(P, label)}")

# --- Class 3 / Class 4 detail ---
print("\n" + "=" * 100)
print("CLASS 3 / CLASS 4 DETAIL — control -> query-lag, per seed")
print("=" * 100)
print(f"  {'seed':>6} {'C3 recall':>19} {'C4 recall':>19} {'3->4':>11} {'4->3':>11}")
for s in done:
    c, r = control[s], rowA[s]
    print(f"  {s:>6} {c['class3_recall']:>8.4f}->{float(r['class3_recall']):<8.4f} {c['class4_recall']:>8.4f}->"
          f"{float(r['class4_recall']):<8.4f} {c['err_3to4']:>4d}->{int(r['err_3to4']):<4d}  {c['err_4to3']:>4d}->{int(r['err_4to3']):<4d}")
print(f"  {'mean':>6} {np.mean([control[s]['class3_recall'] for s in done]):>8.4f}->"
      f"{np.mean([float(rowA[s]['class3_recall']) for s in done]):<8.4f} "
      f"{np.mean([control[s]['class4_recall'] for s in done]):>8.4f}->{np.mean([float(rowA[s]['class4_recall']) for s in done]):<8.4f} "
      f"{np.mean([control[s]['err_3to4'] for s in done]):>5.1f}->{np.mean([float(rowA[s]['err_3to4']) for s in done]):<5.1f} "
      f"{np.mean([control[s]['err_4to3'] for s in done]):>5.1f}->{np.mean([float(rowA[s]['err_4to3']) for s in done]):<5.1f}")

# --- learned mu / sigma ---
print("\n" + "=" * 100)
print(f"LEARNED mu / sigma — final epoch {EPOCHS}, every test-set query token (normalized window time; 1.0 = "
      f"{WINDOW_SEC * 1000:.0f} ms)")
print("=" * 100)
for key, _, label in LACA_DIRECTIONS:
    print(f"  {label}")
    print(f"    {'seed':>6} {'mu mean':>9} {'mu std':>8} {'mu min':>8} {'mu max':>8} {'|mu| ms':>8} {'sat %':>6} "
          f"{'sig mean':>9} {'sig std':>8} {'sig min':>8} {'sig max':>8} {'floor %':>8}")
    for s in done:
        r = rowA[s]
        g = lambda k: float(r[f"final_{key}_{k}"])
        print(f"    {s:>6} {g('mu_mean'):>+9.4f} {g('mu_std'):>8.4f} {g('mu_min'):>+8.4f} {g('mu_max'):>+8.4f} "
              f"{g('mu_absmean') * WINDOW_SEC * 1000:>8.0f} {g('mu_sat_frac') * 100:>6.1f} {g('sigma_mean'):>9.4f} "
              f"{g('sigma_std'):>8.4f} {g('sigma_min'):>8.4f} {g('sigma_max'):>8.4f} {g('sigma_floor_frac') * 100:>8.1f}")
    absmeans = np.array([float(rowA[s][f"final_{key}_mu_absmean"]) for s in done])
    stds = np.array([float(rowA[s][f"final_{key}_mu_std"]) for s in done])
    sig = np.array([float(rowA[s][f"final_{key}_sigma_mean"]) for s in done])
    init_sig = np.array([float(rowA[s][f"init_{key}_sigma_mean"]) for s in done])
    if absmeans.max() < 0.02 and stds.max() < 0.01:
        mu_read = "mu stayed ~0 with no spread on every seed -> no preferred lag and no query dependence learned"
    elif stds.max() < 0.01:
        mu_read = "mu has almost no spread across tokens -> a single shared lag (no query dependence)"
    else:
        ratio = absmeans.max() / max(absmeans.min(), 1e-9)
        mu_read = (f"mu varies across tokens (std {stds.min():.3f}-{stds.max():.3f}) -> query-dependent lags were learned; "
                   f"mean |mu| {absmeans.min():.3f}-{absmeans.max():.3f} across seeds "
                   + ("(consistent)" if ratio <= 3 else f"(seed-dependent, {ratio:.1f}x)"))
    print(f"    -> {mu_read}")
    print(f"    -> sigma: init {init_sig.mean():.3f} -> final {sig.min():.3f}-{sig.max():.3f} across seeds "
          + ("(narrowed: sharper lag preference)" if sig.max() < init_sig.mean() * 0.8 else
             "(widened: flatter, weaker lag preference)" if sig.min() > init_sig.mean() * 1.2 else "(about unchanged)"))

Part A runs finished: 5/5 -> seeds [42, 123, 2024, 7, 99] (each paired with its REUSED v18 beta=0.0 control)

PAIRED — macro-F1: query-dependent lag LACA minus REUSED v18 beta=0.0 control, 5 matched seeds
    seed   control  query-lag      diff
      42    0.9057     0.9075   +0.0018
     123    0.8827     0.8803   -0.0023
    2024    0.8764     0.8782   +0.0018
       7    0.9019     0.9019   -0.0000
      99    0.8638     0.8600   -0.0038
  mean paired difference -0.0005 | std of paired differences 0.0025 | query-lag better on 2/5 (worse 3, ties 0)
  paired t-test (PRIMARY) two-sided p = 0.6740 | Wilcoxon signed-rank p = 0.6250 (smallest achievable with n=5: 0.0625)
  VERDICT (macro-F1): NOT WELL-SUPPORTED

PAIRED — Class 4 recall: query-dependent lag LACA minus REUSED v18 beta=0.0 control, 5 matched seeds
    seed   control  query-lag      diff
      42    0.8507     0.8408   -0.0100
     123    0.7164     0.7512   +0.0348
    2024    0.7463     0.7413   -0.0050
       7    0.8458  

## 8. Part B — timestamp-shuffle control (same trained models, no retraining)
**The shuffle.** For every test window the 128 sensor timestamps τ_s are permuted among themselves. The permutation is independent per window and drawn from a private generator (seed 2718), so the global RNG is untouched. The sensor values, audio, τ_a and labels are unchanged. The first cell of the code checks this: the sensor values are **bit-identical**, and every window keeps the same set of timestamps in a new order.

**Evaluation.** The 5 Part A checkpoints are reloaded, with no training. Each is first confirmed to reproduce its Part A result exactly on the true timestamps, then evaluated in two scopes:
- **all**: the shuffled τ_s is used everywhere, including HTT's time embedding. This is the requested test.
- **LACA only**: HTT sees the true τ_s, and only the new lag term sees the shuffled one. This isolates the new mechanism.

**Degradation** = result with true timestamps − result with shuffled timestamps, for macro-F1 and Class 4 recall. The model **relies on temporal correspondence** if the degradation is positive on at least 4/5 seeds, larger than its own std, and material: at least 0.005 macro-F1 (about 6 windows) or 0.01 Class 4 recall (2 windows).

**The control.** The control also uses τ_s in HTT. If its checkpoints are attached (`CONTROL_CKPT_GLOBS`), it gets the same "all" shuffle. The difference between the two degradations is what the new mechanism adds beyond HTT. Results go to `query_lag_laca_shuffle_results.csv`.

In [11]:
# ==============================================================================
# PART B — timestamp-shuffle control on the SAME Part A models (no retraining)
# ==============================================================================
import glob


class ShuffledSensorTauTestSet(Dataset):
    """The test set with an extra field tau_s_shuffled = tau_s permuted within the window. Nothing else changes."""

    def __init__(self, base, perms):
        self.base, self.perms = base, perms

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        item = dict(self.base[idx])
        item["tau_s_shuffled"] = item["tau_s"][self.perms[idx]]
        return item


_gen = torch.Generator().manual_seed(SHUFFLE_SEED)   # private: the global RNG is not touched
SHUFFLE_PERMS = torch.stack([torch.randperm(TS_SENSOR, generator=_gen) for _ in range(len(test_ds))])
shuf_ds = ShuffledSensorTauTestSet(test_ds, SHUFFLE_PERMS)
shuf_loader = DataLoader(shuf_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)

# --- proof: only the timestamps' order changes ---
orig_items = [test_ds[i] for i in range(len(test_ds))]
shuf_items = [shuf_ds[i] for i in range(len(shuf_ds))]
xs_before = torch.stack([it["x_s"] for it in orig_items])
xs_after = torch.stack([it["x_s"] for it in shuf_items])
tau_before = torch.stack([it["tau_s"] for it in orig_items])
tau_after = torch.stack([it["tau_s_shuffled"] for it in shuf_items])
sha = lambda t: hashlib.sha1(t.contiguous().numpy().tobytes()).hexdigest()[:16]
assert torch.equal(xs_before, xs_after), "sensor VALUES changed"
assert all(torch.equal(o[k], s[k]) for o, s in zip(orig_items, shuf_items) for k in ("x_a", "tau_a", "tau_s", "label"))
assert torch.equal(tau_before.sort(dim=1).values, tau_after.sort(dim=1).values), "a window's set of timestamps changed"
moved = float((tau_before != tau_after).float().mean())
print(f"[SHUFFLE CHECK] sensor values x_s {tuple(xs_before.shape)}: SHA1 before {sha(xs_before)} | after {sha(xs_after)} "
      f"-> bit-identical ✅ (torch.equal) | audio, tau_a, labels, and the original tau_s field: identical ✅")
print(f"[SHUFFLE CHECK] tau_s: every one of the {len(test_ds)} windows has the same SET of timestamps (sorted values "
      f"identical ✅); {moved:.1%} of positions now carry a different timestamp")
print(f"[SHUFFLE CHECK] example, test window 0, first 8 sensor positions (normalized time -> ms):")
print("    before: " + ", ".join(f"{float(v):.4f} ({float(v) * WINDOW_SEC * 1000:.0f} ms)" for v in tau_before[0, :8]))
print("    after : " + ", ".join(f"{float(v):.4f} ({float(v) * WINDOW_SEC * 1000:.0f} ms)" for v in tau_after[0, :8]))
print(f"    window 0 sorted before == sorted after: {torch.equal(tau_before[0].sort().values, tau_after[0].sort().values)} "
      f"| x_s[0] identical: {torch.equal(xs_before[0], xs_after[0])}")
del orig_items, shuf_items, xs_before, xs_after

# --- evaluate the 5 Part A models: correct / all / laca_only ---
if SHUFFLE_CSV.exists():
    SHUFFLE_CSV.rename(SHUFFLE_CSV.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))
SHUFFLE_FIELDS = ["model", "seed", "mode", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc", "class3_recall",
                  "class4_recall", "err_3to4", "err_4to3", "confusion_matrix"]


def log_shuffle(model_name, seed, mode, ev):
    write_header = not SHUFFLE_CSV.exists()
    with open(SHUFFLE_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=SHUFFLE_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow({"model": model_name, "seed": seed, "mode": mode, **{k: ev[k] for k in SHUFFLE_FIELDS[3:-1]},
                         "confusion_matrix": json.dumps(np.asarray(ev["confusion_matrix"]).tolist())})


shuffle_results = {}
for seed in SEEDS:
    model = AeroSyncMamba(lag_mode="query").to(device)
    model.load_state_dict(torch.load(CHECKPOINT_DIR / f"query_lag_seed{seed}_best.pt", map_location=device))
    res = {}
    for mode in ("correct", "all", "laca_only"):
        probs, labels, _ = predict_probs_tau(model, shuf_loader, mode)
        assert np.array_equal(labels, TEST_LABELS)
        res[mode] = evaluate_probs(labels, probs)
        log_shuffle("query_lag", seed, mode, res[mode])
    worst = max(abs(res["correct"][m] - run_results[seed][m]) for m in METRICS)
    shuffle_results[seed] = res
    print(f"\n[PART B seed={seed}] reloaded Part A checkpoint (no retraining) on true timestamps vs Part A: max |delta| "
          f"{worst:.1e} " + ("✅ reproduces Part A exactly" if worst < 1e-9 else
                             "⚠️ tiny difference (within tolerance)" if worst <= SAME_SEED_TOLERANCE else
                             "🚨 does NOT reproduce Part A — check the checkpoint"))
    for mode, label in (("correct", "true timestamps"), ("all", "shuffled tau_s everywhere"), ("laca_only", "shuffled tau_s in LACA only")):
        e = res[mode]
        print(f"    {label:<30} accuracy {e['accuracy']:.4f} | macro-F1 {e['macro_f1']:.4f} | Class 4 recall "
              f"{e['class4_recall']:.4f}" + ("" if mode == "correct" else
              f" | degradation: macro-F1 {res['correct']['macro_f1'] - e['macro_f1']:+.4f}, Class 4 recall "
              f"{res['correct']['class4_recall'] - e['class4_recall']:+.4f}"))
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- optional: the control, shuffled the same way (needs its checkpoints) ---
control_shuffle = {}
for seed in SEEDS:
    hits = [h for p in CONTROL_CKPT_GLOBS for h in sorted(glob.glob(p.format(seed=seed), recursive=True))]
    for h in hits:
        cmodel = AeroSyncMamba(lag_mode="none").to(device)
        missing, unexpected = cmodel.load_state_dict(torch.load(h, map_location=device), strict=False)
        if missing or not all(".lag_bias." in k for k in unexpected):
            print(f"[CONTROL SHUFFLE seed={seed}] {h}: incompatible keys — ignored")
            continue
        r = {mode: evaluate_probs(TEST_LABELS, predict_probs_tau(cmodel, shuf_loader, mode)[0]) for mode in ("correct", "all")}
        worst = max(abs(r["correct"][m] - V18_BETA0_CONTROL[seed][m]) for m in METRICS)
        del cmodel
        if worst > SAME_SEED_TOLERANCE:
            print(f"[CONTROL SHUFFLE seed={seed}] {h}: does not reproduce the control ({worst:.4f}) — ignored")
            continue
        control_shuffle[seed] = r
        for mode in r:
            log_shuffle("control", seed, mode, r[mode])
        print(f"[CONTROL SHUFFLE seed={seed}] {h}: reproduces the control ✅ | shuffled tau_s everywhere: macro-F1 "
              f"degradation {r['correct']['macro_f1'] - r['all']['macro_f1']:+.4f}, Class 4 recall "
              f"{r['correct']['class4_recall'] - r['all']['class4_recall']:+.4f}")
        break
if not control_shuffle:
    print("\n[CONTROL SHUFFLE] control checkpoints not attached -> the control could not be shuffled. The 'all' degradation "
          "below therefore mixes HTT's time embedding with the new lag term; the 'LACA only' degradation isolates the "
          "new mechanism.")

# --- degradation statistics + verdicts ---
def degradation_stats(values):
    v = np.asarray(values, dtype=float)
    k = len(v)
    sd = float(v.std(ddof=1)) if k > 1 else float("nan")
    p = float(stats.ttest_1samp(v, 0.0).pvalue) if k > 1 and sd > 0 else float("nan")
    return {"v": v, "mean": float(v.mean()), "sd": sd, "pos": int((v > 0).sum()), "n": k, "p": p}


def reliance(D, material):
    if D["pos"] >= D["n"] - 1 and D["mean"] >= material and D["mean"] > D["sd"]:
        return "RELIES ON TEMPORAL CORRESPONDENCE (substantial, consistent degradation)"
    if abs(D["mean"]) < material or D["pos"] <= D["n"] // 2:
        return "NOT MEANINGFULLY USING the timestamps (performance ~unchanged / inconsistent under shuffling)"
    return "WEAK / INCONSISTENT reliance (degradation present but not consistent or not beyond its own noise)"


print("\n" + "#" * 110)
print("#  PART B — DEGRADATION = true-timestamp result minus shuffled-timestamp result (positive = shuffling hurts)")
print("#" * 110)
PB = {}
for mode, label in (("all", "shuffled tau_s everywhere (HTT + LACA)"), ("laca_only", "shuffled tau_s in the LACA lag term only")):
    for q, qlabel, material in (("macro_f1", "macro-F1", SHUFFLE_MATERIAL_F1), ("class4_recall", "Class 4 recall", SHUFFLE_MATERIAL_C4)):
        D = degradation_stats([shuffle_results[s]["correct"][q] - shuffle_results[s][mode][q] for s in SEEDS])
        PB[(mode, q)] = D
        print(f"#  {label} — {qlabel}: " + ", ".join(f"{s}: {x:+.4f}" for s, x in zip(SEEDS, D["v"]))
              + f" | mean {D['mean']:+.4f}, std {D['sd']:.4f}, positive on {D['pos']}/{D['n']}, t-test p {D['p']:.4f}")
        print(f"#    -> {reliance(D, material)}")
if control_shuffle:
    cs = [s for s in SEEDS if s in control_shuffle]
    for q, qlabel in (("macro_f1", "macro-F1"), ("class4_recall", "Class 4 recall")):
        dn = np.array([shuffle_results[s]["correct"][q] - shuffle_results[s]["all"][q] for s in cs])
        dc = np.array([control_shuffle[s]["correct"][q] - control_shuffle[s]["all"][q] for s in cs])
        DD = degradation_stats(dn - dc)
        print(f"#  DIFFERENCE-IN-DIFFERENCES ({qlabel}, seeds {cs}): query-lag degradation {dn.mean():+.4f} vs control "
              f"degradation {dc.mean():+.4f} -> extra reliance from the new mechanism {DD['mean']:+.4f} (std {DD['sd']:.4f}, "
              f"positive on {DD['pos']}/{DD['n']})")
print("#" * 110)

[SHUFFLE CHECK] sensor values x_s (1130, 128, 6): SHA1 before e1ca9cdbe6b788be | after e1ca9cdbe6b788be -> bit-identical ✅ (torch.equal) | audio, tau_a, labels, and the original tau_s field: identical ✅
[SHUFFLE CHECK] tau_s: every one of the 1130 windows has the same SET of timestamps (sorted values identical ✅); 99.2% of positions now carry a different timestamp
[SHUFFLE CHECK] example, test window 0, first 8 sensor positions (normalized time -> ms):
    before: 0.0000 (0 ms), 0.0046 (12 ms), 0.0139 (36 ms), 0.0208 (53 ms), 0.0309 (79 ms), 0.0390 (100 ms), 0.0441 (113 ms), 0.0549 (141 ms)
    after : 0.9135 (2339 ms), 0.5894 (1509 ms), 0.2539 (650 ms), 0.0549 (141 ms), 0.1023 (262 ms), 0.9920 (2539 ms), 0.4626 (1184 ms), 0.5775 (1478 ms)
    window 0 sorted before == sorted after: True | x_s[0] identical: True

[PART B seed=42] reloaded Part A checkpoint (no retraining) on true timestamps vs Part A: max |delta| 0.0e+00 ✅ reproduces Part A exactly
    true timestamps                ac

## 9. Final validation and overall reading
A recap of every check: data and RNG diagnostics, control reuse, pairing, the μ/σ initialization check, the NaN/Inf check, and the shuffle checks. It ends with a plain reading that combines Part A (does it help?) and Part B (does it use time?).

In [12]:
# ==============================================================================
# FINAL VALIDATION — every verification hook in one place + the combined reading
# ==============================================================================
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)
data_checks = {
    "strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
    **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
       REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")},
    "seed-42 torch RNG after construction == v16/v18": run_results[42]["rng_sha1"].startswith(REFERENCE_SEED42_RNG_PREFIX),
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in data_checks.items():
    print(f"  {name:<58}: {'match' if ok else 'MISMATCH'}")
print("\n" + "=" * 100)
print(f"  control: v18 beta=0.0 runs REUSED, not retrained : ✅ confusion matrices reproduce v18 exactly for all "
      f"{len(SEEDS)} seeds (macro-F1 " + ", ".join(f"{V18_BETA0_CONTROL[s]['macro_f1']:.4f}" for s in SEEDS) + ")")
print(f"  every run starts where its control started        : "
      + ("✅ all seeds (RNG state after construction identical)" if all(r["paired_with_control"] for r in run_results.values())
         else "🚨 " + str({s: r["paired_with_control"] for s, r in run_results.items()})))
print(f"  mu / sigma at init (not saturated / not extreme)  : ✅ checked on a real batch for every seed (see each run)")
print(f"  NaN / Inf in attention scores                     : never triggered — "
      + ", ".join(f"{s}: {r['finite_checks']:,} checks" for s, r in run_results.items()))
print(f"  confusion matrices printed                        : {len(run_results)} Part A runs + {len(control)} controls")
print(f"  Part B: sensor values bit-identical, same tau set : ✅ | reloaded models vs Part A (no retraining): max |delta| "
      f"{max(abs(shuffle_results[s]['correct'][m] - run_results[s][m]) for s in SEEDS for m in METRICS):.1e}")
print("=" * 100)

f1_v, c4_v = verdict(PA["macro_f1"], "macro-F1"), verdict(PA["class4_recall"], "Class 4 recall")
laca_used = reliance(PB[("laca_only", "macro_f1")], SHUFFLE_MATERIAL_F1)
print("\n" + "#" * 100)
print(f"#  PART A — macro-F1       : {f1_v} (mean {PA['macro_f1']['mean']:+.4f}, std {PA['macro_f1']['sd']:.4f}, wins "
      f"{PA['macro_f1']['wins']}/{PA['macro_f1']['n']}, p {PA['macro_f1']['p_t']:.4f})")
print(f"#  PART A — Class 4 recall : {c4_v} (mean {PA['class4_recall']['mean']:+.4f}, std {PA['class4_recall']['sd']:.4f}, "
      f"wins {PA['class4_recall']['wins']}/{PA['class4_recall']['n']}, p {PA['class4_recall']['p_t']:.4f})")
print(f"#  PART B — new lag term    : {laca_used}")
helps = f1_v.startswith("WELL") or c4_v.startswith("WELL")
uses_time = laca_used.startswith("RELIES")
if helps and uses_time:
    print("#  READING: the query-dependent mechanism helps AND the gain depends on correct timestamps -> evidence of genuine "
          "temporal reasoning.")
elif helps:
    print("#  READING: the architecture helps, but scrambling the timestamps its lag term sees barely matters -> the gain is "
          "most likely NOT temporal reasoning (e.g. extra capacity or a changed attention prior).")
elif uses_time:
    print("#  READING: the model does use the timestamps through the new lag term, but doing so does not improve accuracy "
          "over the control.")
else:
    print("#  READING: no reliable gain, and the new lag term is not meaningfully used -> like the global beta term, "
          "query-dependent lags give no evidence of benefit here.")
print("#" * 100)
if not all(data_checks.values()):
    print("🚨 a data / RNG diagnostic is a MISMATCH — the reused control may not be a valid baseline.")

FINAL VALIDATION
  strongest-64 bin indices                                  : match
  train windows / sensor values / timestamps / class counts : match
  val windows / sensor values / timestamps / class counts   : match
  test windows / sensor values / timestamps / class counts  : match
  seed-42 torch RNG after construction == v16/v18           : match
  fusion backbone is real mamba_ssm                         : match

  control: v18 beta=0.0 runs REUSED, not retrained : ✅ confusion matrices reproduce v18 exactly for all 5 seeds (macro-F1 0.9057, 0.8827, 0.8764, 0.9019, 0.8638)
  every run starts where its control started        : ✅ all seeds (RNG state after construction identical)
  mu / sigma at init (not saturated / not extreme)  : ✅ checked on a real batch for every seed (see each run)
  NaN / Inf in attention scores                     : never triggered — 42: 7,586 checks, 123: 7,586 checks, 2024: 7,586 checks, 7: 7,586 checks, 99: 7,586 checks
  confusion matrices printed    